In [ ]:
# 사전설치 라이브러리 : pip install langgraph langchain-google-genai

# === 1. 필수 라이브러리 설치 및 불러오기 ===
# Langchain과 관련된 핵심 라이브러리들을 불러옵니다.
# - langchain_google_genai: Google의 Gemini 모델을 사용하기 위한 라이브러리
# - langgraph: 에이전트 워크플로우를 그래프 형태로 구축하기 위한 라이브러리
# - langchain_core: Langchain의 기본 데이터 구조(메시지 등)를 다루는 라이브러리
# - typing: 코드의 타입을 명시하여 가독성과 안정성을 높이기 위한 라이브러리
import os
from typing import List, TypedDict

from langchain_core.messages import BaseMessage, HumanMessage
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.graph import END, StateGraph
import gradio as gr

# === 2. Google API 키 설정 ===
os.environ["GOOGLE_API_KEY"] = "insert your key"  # Google AI Studio API 키

# === 3. LLM (언어 모델) 초기화 ===
llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash", temperature=0.7)

c:\aisource\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# === 4. 그래프의 상태(State) 정의 ===
# State는 그래프의 모든 노드(에이전트)가 공유하는 "메모리" 또는 "데이터 저장소"입니다.
# 각 에이전트는 이 State를 읽고, 자신의 작업 결과를 State에 추가(업데이트)하여 다음 에이전트에게 전달합니다.
# TypedDict를 사용하면 State의 구조를 명확하게 정의할 수 있어 실수를 줄여줍니다.
class GraphState(TypedDict):
    topic: str         # [입력] 이야기의 주제
    plan: str          # [기획자]가 생성한 이야기 개요
    draft: str         # [작가]가 작성한 초고
    critique: str      # [비평가]가 작성한 피드백
    review_count: int  # [비평가]가 수정한 횟수 (무한 루프 방지용)
    logs: List[str]    # 각 단계의 진행 상황을 기록하는 로그

In [ ]:
# === 5. 에이전트 노드(Node) 함수 정의 ===
# 각 에이전트는 그래프에서 하나의 '노드(Node)'가 되며, 특정 작업을 수행하는 함수로 정의됩니다.
# 모든 노드 함수는 'state'를 입력으로 받고, 업데이트된 'state'의 일부를 딕셔너리 형태로 반환합니다.

def planner_node(state: GraphState) -> GraphState:
    """기획자 에이전트: 주제에 대한 개요를 생성합니다."""
    print("--- 노드 실행: 기획자 ---")

    # 프롬프트 엔지니어링: LLM에게 명확한 역할과 지시사항을 제공합니다.
    prompt = f"""당신은 창의적인 이야기 기획자입니다.
주제: "{state['topic']}"
이 주제에 대한 흥미로운 이야기의 개요(줄거리)를 시작, 중간, 끝 구조로 작성해주세요."""

    # LLM을 호출하여 결과를 받습니다.
    response = llm.invoke(prompt)
    plan = response.content
    print(f"생성된 개요: {plan[:150]}...")

    # 상태 업데이트: 이 노드의 결과('plan')를 State에 추가하여 반환합니다.
    return {"plan": plan}

def writer_node(state: GraphState) -> GraphState:
    """작가 에이전트: 개요를 바탕으로 초고를 작성하거나, 비평을 바탕으로 수정합니다."""
    print("--- 노드 실행: 작가 ---")

    # State에서 필요한 정보를 가져옵니다. 비평(critique)은 처음에는 없을 수 있습니다.
    plan = state["plan"]
    critique = state.get("critique")

    # 조건부 프롬프트: 비평가의 피드백이 있는지 여부에 따라 다른 프롬프트를 사용합니다.
    if critique:
        # 피드백이 있으면, 수정을 요청하는 프롬프트를 사용합니다.
        print("피드백을 바탕으로 초고를 수정합니다.")
        prompt = f"""당신은 뛰어난 소설가입니다.
아래의 개요와 비평을 바탕으로 이야기 초고를 '수정'해주세요. 이전 초고의 단점을 보완해야 합니다.
개요: {plan}
이전 초고: {state['draft']}
비평: {critique}
수정된 새로운 초고를 작성해주세요."""
    else:
        # 피드백이 없으면, 첫 초고 작성을 요청하는 프롬프트를 사용합니다.
        print("개요를 바탕으로 새 초고를 작성합니다.")
        prompt = f"""당신은 뛰어난 소설가입니다.
아래의 개요를 바탕으로 짧은 이야기를 작성해주세요.
개요: {plan}"""

    response = llm.invoke(prompt)
    draft = response.content
    print(f"작성된 초고: {draft[:150]}...")

    # 상태 업데이트: 이 노드의 결과('draft')를 State에 추가하여 반환합니다.
    return {"draft": draft}

def critic_node(state: GraphState) -> GraphState:
    """비평가 에이전트: 초고를 검토하고 피드백을 제공하며, 다음 단계를 결정합니다."""
    print("--- 노드 실행: 비평가 ---")

    # 프롬프트 엔지니어링: LLM이 명확한 형식으로 답변하도록 유도하는 것이 중요합니다.
    # 'APPROVE' 또는 'REVISE'를 반드시 포함하도록 지시하여, 다음 단계(엣지)를 결정하기 쉽게 만듭니다.
    prompt = f"""당신은 날카로운 문학 비평가입니다.
아래의 이야기 초고를 검토하고 건설적인 피드백을 제공해주세요.
초고: {state['draft']}

검토 후, 다음 중 하나를 결정하여 응답의 '가장 마지막 줄'에 다음 단어 중 하나만 '반드시' 포함시켜주세요:
- 수정이 필요하면: REVISE
- 이야기가 훌륭하면: APPROVE
"""
    response = llm.invoke(prompt)
    critique = response.content
    print(f"생성된 비평: {critique}")

    # 수정 횟수를 1 증가시킵니다. 이는 무한 루프를 방지하는 데 사용됩니다.
    review_count = state.get("review_count", 0) + 1

    # 상태 업데이트: 비평 내용과 수정 횟수를 State에 추가합니다.
    return {"critique": critique, "review_count": review_count}

In [ ]:
# === 6. 조건부 엣지(Edge) 로직 정의 ===
# 엣지는 노드와 노드를 연결하는 '경로'입니다.
# '조건부 엣지'는 특정 조건에 따라 다음 경로를 동적으로 결정하게 해줍니다.
# 여기서는 비평가의 결과에 따라 '작업을 종료할지' 또는 '작가에게 수정을 요청할지' 결정합니다.

def should_continue(state: GraphState) -> str:
    """비평가의 피드백을 분석하여 다음 단계를 결정하는 분기점 역할을 합니다."""
    print("--- 조건부 엣지 평가 ---")
    critique = state["critique"]
    review_count = state["review_count"]

    # 안전장치 1: 최대 수정 횟수 제한 (무한 루프 방지)
    if review_count > 3:
        print("결정: 최대 수정 횟수 도달. 작업을 종료합니다.")
        return "end"

    # 안전장치 2: LLM이 'approve' 또는 'Approve' 등 다양하게 출력할 수 있으므로,
    # 대문자로 변환하여 일관성 있게 비교합니다.
    if "APPROVE" in critique.upper():
        print("결정: 승인(APPROVE). 작업을 종료합니다.")
        return "end"  # 'end'를 반환하면 그래프 실행이 종료됩니다.
    else:
        print("결정: 수정 필요(REVISE). 작가 노드로 돌아갑니다.")
        return "revise" # 'revise'를 반환하면 작가 노드로 가는 엣지와 연결됩니다.

In [ ]:
# === 7. 그래프 구축 및 컴파일 ===
# 이제 정의한 State, 노드, 엣지를 연결하여 실제 워크플로우 그래프를 만듭니다.

# 1. StateGraph 객체를 생성하고, 우리가 정의한 State 구조를 알려줍니다.
workflow = StateGraph(GraphState)

# 2. 정의한 함수들을 그래프의 노드로 추가합니다.
workflow.add_node("planner", planner_node)
workflow.add_node("writer", writer_node)
workflow.add_node("critic", critic_node)

# 3. 노드 간의 엣지(연결)를 설정합니다.
workflow.set_entry_point("planner") # 'planner' 노드에서 그래프 실행을 시작합니다.
workflow.add_edge("planner", "writer")    # 'planner'가 끝나면 항상 'writer'로 갑니다.
workflow.add_edge("writer", "critic")     # 'writer'가 끝나면 항상 'critic'으로 갑니다.

# 4. 'critic' 노드 다음에 실행될 조건부 엣지를 추가합니다.
workflow.add_conditional_edges(
    "critic",         # 'critic' 노드에서 분기가 시작됩니다.
    should_continue,  # 'should_continue' 함수의 반환값에 따라 다음 경로를 결정합니다.
    {
        "revise": "writer", # 'revise'를 반환하면 'writer' 노드로 다시 돌아갑니다 (피드백 루프 생성).
        "end": END          # 'end'를 반환하면 그래프 실행을 완전히 종료합니다 (END는 langgraph의 특별한 상수).
    }
)

# 5. 그래프 정의를 완료하고 실행 가능한 객체로 컴파일합니다.
app = workflow.compile()

In [ ]:
# === 8. Gradio 인터페이스를 위한 실행 함수 정의 ===
# Gradio는 이 함수를 호출하여 LangGraph 워크플로우를 실행하고 결과를 UI에 업데이트합니다.
# stream()의 출력을 실시간으로 UI에 반영하기 위해 제너레이터(yield)를 사용합니다.
def run_graph_for_gradio(topic: str):
    if not topic:
        yield "주제를 입력해주세요.", "", "", ""
        return

    # 초기 상태 설정
    inputs = {"topic": topic}
    config = {"recursion_limit": 15}

    progress_log = [] # 진행 상황을 저장할 리스트
    final_state = {}

    # app.stream()을 사용하여 각 노드의 결과를 실시간으로 받습니다.
    for output in app.stream(inputs, config=config):
        # output 딕셔너리에서 마지막으로 실행된 노드의 이름을 찾습니다.
        last_node = list(output.keys())[-1]
        node_output = output[last_node]

        # 각 노드의 결과물을 로그에 추가합니다.
        log_entry = f"### 노드 '{last_node}' 실행 완료\n\n"
        if last_node == 'planner':
            log_entry += f"**생성된 개요:**\n\n---\n{node_output['plan']}\n---"
        elif last_node == 'writer':
            log_entry += f"**작성된 초고:**\n\n---\n{node_output['draft']}\n---"
        elif last_node == 'critic':
            log_entry += f"**생성된 비평:**\n\n---\n{node_output['critique']}\n---"

        progress_log.append(log_entry)

        # 지금까지의 로그를 하나의 문자열로 합칩니다.
        log_so_far = "\n\n".join(progress_log)

        # Gradio UI를 업데이트합니다. 아직 최종 결과는 없으므로 빈 문자열을 전달합니다.
        yield log_so_far, "", "", ""

        # 마지막 상태를 계속 저장해 둡니다.
        final_state = node_output

    # 루프가 끝나면 최종 결과물을 준비합니다.
    final_plan = final_state.get('plan', '결과 없음')
    final_draft = final_state.get('draft', '결과 없음')
    final_critique = final_state.get('critique', '결과 없음')

    # 최종 결과물로 UI를 마지막으로 한 번 더 업데이트합니다.
    yield log_so_far, final_plan, final_draft, final_critique

In [ ]:
# === 9. Gradio UI 블록 정의 ===
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# LangGraph 협동 창작 글쓰기 팀")
    gr.Markdown("이야기 주제를 입력하면, 3명의 AI 에이전트(기획자, 작가, 비평가)가 협력하여 짧은 이야기를 만듭니다.")

    with gr.Row():
        topic_input = gr.Textbox(label="이야기 주제", placeholder="예: 화성에서 길을 잃은 로봇의 이야기", scale=4)
        run_button = gr.Button("이야기 생성 시작", variant="primary", scale=1)

    gr.Markdown("---")

    # 1. 진행 과정 출력 영역을 Textbox로 변경하고 높이 30줄을 설정합니다.
    progress_display = gr.Textbox(
        label="진행 과정",
        lines=30,              # 최소 30줄 높이로 설정
        autoscroll=True,       # 새로운 텍스트가 추가될 때 자동으로 스크롤
        interactive=False,     # 사용자가 직접 입력하지 못하도록 설정
        value="시작 버튼을 눌러주세요."
    )

    with gr.Accordion("최종 결과물 보기", open=False):
        with gr.Row():
            # 2. 최종 개요 출력 영역을 Textbox로 변경하고 높이 30줄을 설정합니다.
            plan_output = gr.Textbox(
                label="최종 개요",
                lines=30,
                interactive=False,
                show_copy_button=True # 텍스트 복사 버튼 추가
            )

            # 3. 최종 초고 출력 영역을 Textbox로 변경하고 높이 30줄을 설정합니다.
            draft_output = gr.Textbox(
                label="최종 초고",
                lines=30,
                interactive=False,
                show_copy_button=True
            )

            # 4. 마지막 비평 출력 영역을 Textbox로 변경하고 높이 30줄을 설정합니다.
            critique_output = gr.Textbox(
                label="마지막 비평",
                lines=30,
                interactive=False,
                show_copy_button=True
            )

    # 버튼 클릭 이벤트를 실행 함수와 연결합니다.
    run_button.click(
        fn=run_graph_for_gradio,
        inputs=[topic_input],
        outputs=[progress_display, plan_output, draft_output, critique_output]
    )

In [ ]:
# === 10. Gradio 앱 실행 ===
demo.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


--- 노드 실행: 기획자 ---
생성된 개요: ## 화성 정착 이야기: 붉은 행성의 푸른 꿈 (이야기 개요)

**주제:** 지구인들이 화성에 정착하는 이야기

**핵심 키워드:** 개척, 생존, 갈등, 희망, 진화

**시작 (Setting the Stage):**

*   **배경:** 2077년, 지구는 심각...
--- 노드 실행: 작가 ---
개요를 바탕으로 새 초고를 작성합니다.
작성된 초고: ## 붉은 행성의 푸른 꿈

2077년, 잿빛 하늘 아래 썩어가는 지구는 신음하고 있었다. 마지막 숨을 헐떡이는 행성을 뒤로하고, 인류는 마지막 희망을 붉은 별, 화성에 걸었다. 식물학자 리아는 그 희망의 씨앗을 품은 사람이었다. 그녀는 멸망해가는 지구를 보며 절망했지...
--- 노드 실행: 비평가 ---
생성된 비평: ## 날카로운 문학 비평

**전반적인 감상:**

이야기의 기본 설정은 흥미롭고 매력적입니다. 멸망한 지구를 떠나 화성에서 새로운 시작을 꿈꾸는 인류, 그리고 그 과정에서 겪는 어려움과 갈등은 독자에게 깊은 인상을 남길 잠재력이 있습니다. 특히 리아라는 캐릭터를 통해 희망과 끈기를 보여주는 점은 긍정적입니다. 그러나 이야기가 전개되는 방식, 캐릭터의 깊이, 그리고 주제 의식의 전달에 있어서 개선할 부분이 있습니다.

**강점:**

*   **매력적인 설정:** 멸망한 지구와 붉은 행성 화성의 대비는 강력한 시각적 이미지를 제공하며, 독자의 호기심을 자극합니다.
*   **주인공의 긍정적인 면모:** 리아는 멸망의 위기 속에서도 희망을 잃지 않고 노력하는 이상적인 인물입니다. 그녀의 헌신적인 모습은 독자에게 감동을 선사할 수 있습니다.
*   **테마의 잠재력:** 인간의 생존 의지, 환경 문제, 공동체의 중요성 등 다양한 주제를 다룰 수 있는 가능성을 보여줍니다.

**개선점:**

1.  **캐릭터의 깊이 부족:** 리아를 제외한 다른 캐릭터들의 개성이 뚜렷하지 않습니다. 이주민들 사이의 갈등은 피상적으로 묘사되어 있으며, 각각의 인물

In [ ]:
demo.close()

Closing server running on port: 7860


생성된 개요: ## 화성에서 길을 잃은 로봇 이야기: 줄거리 개요

**주제:** 화성에서 길을 잃은 로봇 이야기

**주요 등장인물:**

*   **러스티 (Rusty):** 인간의 감정을 학습하고 모방하도록 설계된 탐사 로봇. 낡고 오래되었지만, 뛰어난 적응력과 긍정적인 성격을...
